# Example 1: 使用Modular Strategy执行一个Job

In [2]:
import os
import numpy as np

from ABQflow import BatchAbaqusProcessor, JobSpec, PreparationSpec, HookSpec

%reload_ext autoreload
%autoreload 2

ABAQUS_CAE = 'C:/Applications/SIMULIA/Commands/abaqus.bat'
CWD = os.getcwd()

## 修改inp, 并从该Job的Odb中提取信息

In [3]:
spec1 = JobSpec(
	job_name = "planar_stress_odb",
	workflow = "modular",
	preparation = PreparationSpec(
		kind = "inp_based",
		source_path = "./examples/SingleParameterizedJob/cae_file/planar_stress_template.inp",
		params = {
			"youngs_modulus": 210000,
			"load_magnitude": 2000,
		}
	),
	post_extraction = [
		HookSpec(
			script_path = "./examples/SingleParameterizedJob/cae_file/get_max_stress_mises.py",
			tasks = [
				{"result_name": "max_stress_mises",},
				{"result_name": "max_displacement",},
			]
		)
	]
)

import pprint
pprint.pprint(spec1)

JobSpec(job_name='planar_stress_odb',
        workflow='modular',
        preparation=PreparationSpec(kind='inp_based',
                                    source_path='./examples/SingleParameterizedJob/cae_file/planar_stress_template.inp',
                                    params={'load_magnitude': 2000,
                                            'youngs_modulus': 210000}),
        monolithic_script=None,
        monolithic_params={},
        pre_extraction=[],
        post_extraction=[HookSpec(script_path='./examples/SingleParameterizedJob/cae_file/get_max_stress_mises.py',
                                  tasks=[{'result_name': 'max_stress_mises'},
                                         {'result_name': 'max_displacement'}])])


In [3]:
processor_odb = BatchAbaqusProcessor(
	batch_data = [spec1],
	base_output_dir = os.path.join(CWD, "examples/SingleParameterizedJob/output"),
	cpus_per_job = 4,
	duplicate_mode = "overwrite",
	abaqus_exe = ABAQUS_CAE,
)

In [4]:
outcomes_odb = processor_odb.run_batch(
	num_parallel_jobs = 4,
)

Output()

In [5]:
from ABQflow import outcomes_to_dict

for oc in outcomes_odb:
	if oc.status == 'COMPLETED':
		print(oc.job_name, oc.results['max_stress_mises'])
		print(oc.job_name, oc.results['max_displacement'])
	else:
		print(f"{oc.job_name} Fail: {oc.error}")

results_by_name = outcomes_to_dict(outcomes_odb)

planar_stress_odb 4525.26025390625
planar_stress_odb 3.9895615577697754


## 修改inp, 并从inp中提取信息

In [6]:
spec2 = JobSpec(
	job_name = "planar_stress_mass",
	workflow = "modular",
	preparation = PreparationSpec(
		kind = "inp_based",
		source_path = "./examples/SingleParameterizedJob/cae_file/planar_stress_template.inp",
		params = {
			"youngs_modulus": 210000,
			"load_magnitude": 2000,
		}
	),
	pre_extraction = [
		HookSpec(
			script_path = "./examples/SingleParameterizedJob/cae_file/get_total_mass.py",
			tasks = [
				{"result_name": "total_mass",},
			]
		)
	]
)

In [7]:
processor_mass = BatchAbaqusProcessor(
	batch_data = [spec2],
	base_output_dir = os.path.join(CWD, "examples/SingleParameterizedJob/output"),
	cpus_per_job = 4,
	duplicate_mode = "overwrite",
	abaqus_exe = ABAQUS_CAE,
)

In [8]:
outcomes_mass = processor_mass.run_batch(
	num_parallel_jobs = 4,
)

Output()

In [9]:
from ABQflow import outcomes_to_list

for oc in outcomes_mass:
	if oc.status == 'COMPLETED':
		print(oc.job_name, oc.results['total_mass'])
	else:
		print(f"{oc.job_name} Fail: {oc.error}")

results_by_name = outcomes_to_list(outcomes_mass)   # 需要 list 形式时

planar_stress_mass 0.00032066262255247625
